# Problem 1 — Denoising the Egyptian Goose with PCA and NMF

We are given a clean and a noisy $512\times512$ greyscale image of an Egyptian goose. For a selection of integers $q$ covering $1,\dots,512$ we (i) apply **PCA** to the noisy image (rows as samples in $\mathbb{R}^{512}$) and reconstruct from the first $q$ principal components, and (ii) apply **NMF**, factorising the noisy image as $WH$ with $W\in\mathbb{R}^{512\times q}_{\ge 0}$, $H\in\mathbb{R}^{q\times 512}_{\ge 0}$, reconstructing via $WH$. Quality is the **Euclidean (Frobenius) distance** to the clean image, $d(A,B)=\sqrt{\sum_{i,j}(A_{ij}-B_{ij})^2}$. A method *denoises* if some $q$ gives $d(X_{\text{clean}},\hat X_q)<d_0$, where $d_0=d(X_{\text{clean}},X_{\text{noisy}})$ is the baseline. The PCA and NMF implementations follow the Week 2 and Week 3 reference notebooks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
from sklearn.decomposition import PCA

plt.style.use("ggplot")

X_clean = np.loadtxt("egyptian_goose_clean.csv", delimiter=",")
X_noisy = np.loadtxt("egyptian_goose_noisy.csv", delimiter=",")

def euclid(A, B):
    """Euclidean (Frobenius) distance between two equal-shaped arrays."""
    return np.sqrt(np.sum((A - B) ** 2))

baseline = euclid(X_clean, X_noisy)
print("clean range [%.3f, %.3f];  noisy range [%.3f, %.3f]"
      % (X_clean.min(), X_clean.max(), X_noisy.min(), X_noisy.max()))
print("Baseline  d(clean, noisy) = %.4f" % baseline)

The noisy image is already strictly positive (range above), so it is directly admissible to NMF and we apply NMF to it without any shift. (The Week 3 example used `0.5*(X+1.01)` only because its digit data lived in $[-1,1]$ with zeros; that is unnecessary here.)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 3))
for ax, img, t in zip(axs, [X_clean, X_noisy], ["Clean", "Noisy"]):
    ax.imshow(img, cmap="gray", vmin=X_clean.min(), vmax=X_clean.max())
    ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()

## Choice of $q$ grid

We let the data guide the grid via a **scree plot** of the noisy image's principal components — the tool used in the Week 2 reference to decide how many components to keep. The variance is heavily front-loaded, so we sample densely at low $q$, refine over the mid-band where most variance is captured, and keep a coarse tail to $q=512$ (included as a sanity check: there the reconstruction must recover the noisy image exactly).

In [ ]:
scree = PCA().fit(X_noisy)
evr = scree.explained_variance_ratio_
cum = np.cumsum(evr)

fig, axs = plt.subplots(1, 2, figsize=(9, 3))
axs[0].bar(range(1, 31), evr[:30] * 100, color="salmon", zorder=2)
axs[0].set_title("Scree plot"); axs[0].set_xlabel("Principal component")
axs[0].set_ylabel("Explained variance (%)")
axs[1].plot(range(1, len(cum) + 1), cum * 100, color="royalblue", zorder=2)
axs[1].set_title("Cumulative explained variance")
axs[1].set_xlabel("Number of components q"); axs[1].set_ylabel("Cumulative (%)")
plt.tight_layout(); plt.show()

print("PC1 %.1f%%, PC2 %.1f%%;  80%% at q=%d, 95%% at q=%d"
      % (evr[0]*100, evr[1]*100,
         np.searchsorted(cum, 0.80)+1, np.searchsorted(cum, 0.95)+1))

q_grid = np.array(sorted(set(
    list(range(1, 21))
    + [25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 90, 100]
    + [110, 125, 150, 175, 200, 225, 250, 275, 300]
    + [350, 400, 450, 500, 512]
)))

## PCA (Week 2)

Treating the $512$ rows as samples in $\mathbb{R}^{512}$, for each $q$ we fit `PCA(n_components=q)`, project with `fit_transform`, and reconstruct with `inverse_transform`. At $q=512$ this reproduces the noisy image exactly.

In [ ]:
pca_dist = np.empty(len(q_grid))
pca_recons = {}
for i, q in enumerate(q_grid):
    pca = PCA(n_components=int(q))
    recon = pca.inverse_transform(pca.fit_transform(X_noisy))
    pca_dist[i] = euclid(X_clean, recon)
    pca_recons[int(q)] = recon

best_pca_idx = int(np.argmin(pca_dist)); best_pca_q = int(q_grid[best_pca_idx])
print("PCA: best q = %d -> d = %.4f  (baseline %.4f)"
      % (best_pca_q, pca_dist[best_pca_idx], baseline))
print("PCA: d at q=512 = %.4f  (matches baseline)" % pca_dist[-1])

## NMF (Week 3)

We use the `NMF_numpy` class from the Week 3 reference **verbatim**. It maximises the quasi-likelihood $L(W,H)=\sum X\odot\log(WH)-WH$ via multiplicative updates; the reconstruction is `.WH`. A convergence check at $q=20$ justifies the iteration count used in the sweep.

In [ ]:
class NMF_numpy:
    def train(self, X: npt.NDArray[np.number], q: int, n_iter: int = 100) -> None:
        d, n = X.shape
        self.W = np.random.uniform(low=0, high=1, size=(d, q)).astype(X.dtype)
        self.H = np.random.uniform(low=0, high=1 / q, size=(q, n)).astype(X.dtype)
        self.WH = self.W @ self.H
        self.logL = np.empty(n_iter, dtype=X.dtype)
        self.dist = np.empty(n_iter, dtype=X.dtype)
        for a in range(n_iter):
            self.W *= ((X / self.WH) @ self.H.T) / self.H.sum(axis=1)[None, :]
            self.WH = self.W @ self.H
            self.H *= (self.W.T @ (X / self.WH)) / self.W.sum(axis=0)[:, None]
            self.WH = self.W @ self.H
            self.logL[a] = np.sum(X * np.log(self.WH) - self.WH)
            self.dist[a] = np.sum((X - self.WH) ** 2)

In [ ]:
np.random.seed(0)
check = NMF_numpy(); check.train(X_noisy, q=20, n_iter=500)
fig, axs = plt.subplots(1, 2, figsize=(8, 2.6))
axs[0].plot(range(1, 501), check.logL, color="red")
axs[0].set_xlabel("Iteration"); axs[0].set_ylabel(r"$L(W,H)$")
axs[1].plot(range(1, 501), check.dist, color="blue")
axs[1].set_xlabel("Iteration"); axs[1].set_ylabel(r"$\|X-WH\|^2$")
plt.tight_layout(); plt.show()

The objective flattens well before 500 iterations, so `n_iter = 400` suffices for the sweep (random $W,H$ initialisation is seeded for reproducibility).

In [ ]:
np.random.seed(0)
nmf_dist = np.empty(len(q_grid)); nmf_recons = {}
for i, q in enumerate(q_grid):
    nmf = NMF_numpy(); nmf.train(X_noisy, int(q), n_iter=400)
    nmf_dist[i] = euclid(X_clean, nmf.WH)
    nmf_recons[int(q)] = nmf.WH

best_nmf_idx = int(np.argmin(nmf_dist)); best_nmf_q = int(q_grid[best_nmf_idx])
print("NMF: best q = %d -> d = %.4f  (baseline %.4f)"
      % (best_nmf_q, nmf_dist[best_nmf_idx], baseline))

## Results

In [ ]:
plt.figure(figsize=(7, 3.4))
plt.plot(q_grid, pca_dist, "-o", color="royalblue", markersize=3, label="PCA")
plt.plot(q_grid, nmf_dist, "-s", color="seagreen", markersize=3, label="NMF")
plt.axhline(baseline, color="black", linestyle="--",
            label="Baseline = %.2f" % baseline)
plt.scatter([best_pca_q], [pca_dist[best_pca_idx]], color="royalblue", s=60, edgecolor="k", zorder=5)
plt.scatter([best_nmf_q], [nmf_dist[best_nmf_idx]], color="seagreen", s=60, edgecolor="k", zorder=5)
plt.xlabel("q"); plt.ylabel("distance to clean image")
plt.title("Distance to clean image vs. q"); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(11, 3))
panels = [("Clean", X_clean), ("Noisy (d=%.2f)" % baseline, X_noisy),
          ("PCA q=%d (d=%.2f)" % (best_pca_q, pca_dist[best_pca_idx]), pca_recons[best_pca_q]),
          ("NMF q=%d (d=%.2f)" % (best_nmf_q, nmf_dist[best_nmf_idx]), nmf_recons[best_nmf_q])]
for ax, (t, img) in zip(axs, panels):
    ax.imshow(np.clip(img, 0, 1), cmap="gray", vmin=X_clean.min(), vmax=X_clean.max())
    ax.set_title(t, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

print("PCA best q=%d  d=%.4f  improvement=%.4f  denoises=%s"
      % (best_pca_q, pca_dist[best_pca_idx], baseline - pca_dist[best_pca_idx],
         pca_dist[best_pca_idx] < baseline))
print("NMF best q=%d  d=%.4f  improvement=%.4f  denoises=%s"
      % (best_nmf_q, nmf_dist[best_nmf_idx], baseline - nmf_dist[best_nmf_idx],
         nmf_dist[best_nmf_idx] < baseline))

## Conclusion

**Yes — both PCA and NMF denoise the image.** For each method there is a wide band of $q$ for which the reconstruction is strictly closer to the clean image than the noisy image is (best $q\approx 80$ for PCA and $q\approx 275$ for NMF, each improving the distance from $29.55$ to $\approx 25.0$).

The distance-vs-$q$ curve is **U-shaped**. Noise spreads its energy across all $512$ directions, while the image's structure is concentrated in the leading components (the scree plot: PC1 alone $\approx 27\%$). For small $q$ the reconstruction under-fits (genuine structure is discarded); at an intermediate $q$ it keeps the signal while rejecting the many noise-dominated high-index components — the denoising optimum; as $q\to 512$ it recovers the noisy image ever more faithfully and the distance climbs back to $d_0$ (for PCA this is exact, verified at $q=512$). PCA reaches its optimum earlier and slightly lower than NMF, consistent with its components being the variance-optimal directions. The improvement is real but moderate ($\approx 15\%$), because the broadband noise can only be partly removed by a low-rank truncation.